# C2 - Ops Alert Messages with LangChain

For each restaurant-zone combination where more than 30% of its orders are high cancel_risk, generate an ops manager alert.

Model note: the spec says Claude (claude-sonnet-4-20250514). Substituted Azure OpenAI gpt-4.1-nano via Foundry because Anthropic access is not available on the free trial. The LangChain PromptTemplate + LCEL chain + evaluation rubric are unchanged - only the LLM backend differs.

## Cell 1 - Install dependencies

In [28]:
!pip install langchain langchain-openai --quiet
print('Dependencies installed')

Dependencies installed


## Cell 2 - Mount Google Drive

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 3 - Config

In [ ]:
SCORED_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/urbaneats_scored.csv"

AZURE_ENDPOINT   = "https://spectraverse-eastus2-resource.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME  = "gpt-5.4-nano"
AZURE_API_KEY    = "Enter your Azure OpenAI API key here"

HIGH_RISK_THRESHOLD = 0.30

print(f'Model      : {DEPLOYMENT_NAME}')
print(f'Threshold  : high_risk_rate > {HIGH_RISK_THRESHOLD}')


Model      : gpt-5.4-nano
Threshold  : high_risk_rate > 0.3


## Cell 4 - Load the scored data from C1

In [31]:
import pandas as pd

scored = pd.read_csv(SCORED_CSV_PATH)
print(f'Rows loaded: {len(scored)}')
print(scored['cancel_risk'].value_counts())
scored.head()

Rows loaded: 150
cancel_risk
low     101
high     49
Name: count, dtype: int64


,order_id,order_date,restaurant_name,delivery_zone,order_value,delivery_time_mins,rider_rating,order_status,payment_method,discount_applied,customer_complaints,restaurant_enc,zone_enc,cancel_probability,cancel_risk
0,ORD00001,2024-09-25,Pizza Palace,North,1705.0,54.5,3.0,Delayed,Cash,12.0,0,1,1,0.25,low
1,ORD00002,2024-03-11,Pizza Palace,East,807.0,53.5,4.2,Cancelled,Card,12.0,3,1,0,0.15,low
2,ORD00003,2024-12-11,Spice Garden,South,466.0,46.0,3.6,Delayed,Card,1.0,3,2,4,0.55,high
3,ORD00004,2024-07-10,Wrap & Roll,East,675.0,53.5,3.6,Delivered,UPI,3.0,0,4,0,0.31,low
4,ORD00005,2024-03-15,Wrap & Roll,North,349.0,54.5,2.0,Refunded,Cash,20.0,0,4,1,0.39,low


## Cell 5 - Step 14: aggregate per restaurant x zone

high_risk_rate = high_risk_orders / total_orders per group, with the supporting averages the prompt will need.

In [32]:
scored['is_high_risk'] = (scored['cancel_risk'] == 'high').astype(int)

grouped = (
    scored.groupby(['restaurant_name', 'delivery_zone'])
          .agg(total_orders      = ('cancel_risk', 'size'),
               high_risk_orders  = ('is_high_risk', 'sum'),
               avg_order_value   = ('order_value', 'mean'),
               avg_delivery_time = ('delivery_time_mins', 'mean'))
          .reset_index()
)
grouped['high_risk_rate'] = grouped['high_risk_orders'] / grouped['total_orders']
grouped = grouped.sort_values('high_risk_rate', ascending=False)
grouped

,restaurant_name,delivery_zone,total_orders,high_risk_orders,avg_order_value,avg_delivery_time,high_risk_rate
0,Burger Hub,Central,10,8,1196.300000,67.300000,0.800000
4,Burger Hub,West,4,3,884.500000,75.000000,0.750000
20,Wrap & Roll,Central,4,3,818.250000,53.500000,0.750000
13,Spice Garden,South,8,5,757.375000,48.000000,0.625000
24,Wrap & Roll,West,5,3,595.400000,63.600000,0.600000
14,Spice Garden,West,7,4,1093.714286,58.000000,0.571429
5,Pizza Palace,Central,2,1,1258.000000,60.000000,0.500000
10,Spice Garden,Central,4,2,861.000000,41.250000,0.500000
15,Sushi Bay,Central,9,4,770.888889,40.444444,0.444444
2,Burger Hub,North,5,2,709.800000,50.200000,0.400000


## Cell 6 - Step 15: filter hotspots where high_risk_rate > 0.30

In [33]:
hotspots = grouped[grouped['high_risk_rate'] > HIGH_RISK_THRESHOLD].copy()
print(f'Hotspots flagged: {len(hotspots)} of {len(grouped)} restaurant-zone combinations')
hotspots

Hotspots flagged: 13 of 25 restaurant-zone combinations


,restaurant_name,delivery_zone,total_orders,high_risk_orders,avg_order_value,avg_delivery_time,high_risk_rate
0,Burger Hub,Central,10,8,1196.300000,67.300000,0.800000
4,Burger Hub,West,4,3,884.500000,75.000000,0.750000
20,Wrap & Roll,Central,4,3,818.250000,53.500000,0.750000
13,Spice Garden,South,8,5,757.375000,48.000000,0.625000
24,Wrap & Roll,West,5,3,595.400000,63.600000,0.600000
14,Spice Garden,West,7,4,1093.714286,58.000000,0.571429
5,Pizza Palace,Central,2,1,1258.000000,60.000000,0.500000
10,Spice Garden,Central,4,2,861.000000,41.250000,0.500000
15,Sushi Bay,Central,9,4,770.888889,40.444444,0.444444
2,Burger Hub,North,5,2,709.800000,50.200000,0.400000


## Cell 7 - Build LangChain chain (PromptTemplate + Azure LLM + parser)

In [34]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    model=DEPLOYMENT_NAME,
    temperature=0.3,
    max_tokens=300,
)

TEMPLATE_V1 = """You are an operations analyst for UrbanEats, a food delivery company.

Restaurant   : {restaurant_name}
Delivery zone: {delivery_zone}
High cancel-risk rate: {high_risk_rate}
Average order value  : {avg_order_value}
Average delivery time: {avg_delivery_time}

Write an ops alert for the regional manager. Follow these rules exactly:
1. Output EXACTLY two sentences. Nothing else. No greeting, no sign-off, no bullet points.
2. Sentence 1 must begin with the literal restaurant name followed by the literal zone name, and must state the high cancel-risk rate as a percentage.
3. Sentence 2 must recommend ONE concrete operational action. Choose from: increase rider allocation, review restaurant prep times, audit kitchen capacity, adjust dispatch routing, or add staff during peak hours.
4. Do NOT use any of these hedging words: seems, perhaps, might, appears, possibly, could be, may.
5. Use plain text only."""

prompt_v1 = PromptTemplate(
    input_variables=['restaurant_name','delivery_zone','high_risk_rate',
                     'avg_order_value','avg_delivery_time'],
    template=TEMPLATE_V1,
)

chain_v1 = prompt_v1 | llm | StrOutputParser()
print('Chain ready')


Chain ready


## Cell 8 - Step 16: generate an alert for every hotspot

In [35]:
from datetime import date
def format_inputs(row):
    return {
        'restaurant_name'  : row['restaurant_name'],
        'delivery_zone'    : row['delivery_zone'],
        'high_risk_rate'   : f"{row['high_risk_rate']*100:.0f}%",
        'avg_order_value'  : f"${row['avg_order_value']:.0f}",
        'avg_delivery_time': f"{row['avg_delivery_time']:.0f} minutes",
    }

alerts = []
for _, row in hotspots.iterrows():
    inputs = format_inputs(row)
    alert  = chain_v1.invoke(inputs).strip()
    alerts.append(alert)
    header = f"{row['restaurant_name']} - {row['delivery_zone']} ({inputs['high_risk_rate']} high risk)"
    print('-' * len(header))
    print(header)
    print('-' * len(header))
    print(alert)
    print()

hotspots['alert'] = alerts
hotspots['generated_at'] = date.today().isoformat()

------------------------------------
Burger Hub - Central (80% high risk)
------------------------------------
Burger Hub Central has a high cancel-risk rate of 80%. Review restaurant prep times to reduce order cancellations by aligning expected prep with the 67-minute average delivery time.

---------------------------------
Burger Hub - West (75% high risk)
---------------------------------
Burger Hub West has a high cancel-risk rate of 75%. Review restaurant prep times.

-------------------------------------
Wrap & Roll - Central (75% high risk)
-------------------------------------
Wrap & Roll Central has a 75% high cancel-risk rate with an average order value of $818 and an average delivery time of 54 minutes. Audit kitchen capacity to eliminate delays that drive cancellations.

------------------------------------
Spice Garden - South (62% high risk)
------------------------------------
Spice Garden South has a high cancel-risk rate of 62% with an average order value of $757 and 

## Cell 9 - Step 17: evaluate 3 alerts on specificity, actionability, no-hedging

In [36]:
import re

ACTION_KEYWORDS = ['rider','prep','dispatch','kitchen','allocate','capacity',
                   'route','staff','peak','audit']
HEDGE_PATTERN   = re.compile(r'\b(seems|perhaps|might|appears|possibly|could be|may)\b',
                              re.IGNORECASE)

def score_alert(alert, restaurant_name, delivery_zone):
    text = alert.lower()
    specificity   = restaurant_name.lower() in text and delivery_zone.lower() in text
    actionability = any(kw in text for kw in ACTION_KEYWORDS)
    no_hedging    = HEDGE_PATTERN.search(alert) is None
    return specificity, actionability, no_hedging

sample = hotspots.head(3).copy()
scores = sample.apply(
    lambda r: score_alert(r['alert'], r['restaurant_name'], r['delivery_zone']),
    axis=1, result_type='expand'
)
scores.columns = ['specificity','actionability','no_hedging']

evaluation = pd.concat([
    sample[['restaurant_name','delivery_zone','alert']].reset_index(drop=True),
    scores.reset_index(drop=True),
], axis=1)
evaluation['all_pass'] = evaluation[['specificity','actionability','no_hedging']].all(axis=1)
evaluation

,restaurant_name,delivery_zone,alert,specificity,actionability,no_hedging,all_pass
0,Burger Hub,Central,Burger Hub Central has a high cancel-risk rate...,True,True,True,True
1,Burger Hub,West,Burger Hub West has a high cancel-risk rate of...,True,True,True,True
2,Wrap & Roll,Central,Wrap & Roll Central has a 75% high cancel-risk...,True,True,True,True


## Cell 10 - Revise prompt if any of the 3 failed

If all_pass is True for all 3 rows above, no revision is needed and we keep the v1 prompt. Otherwise we tighten the failing dimension in a v2 prompt and re-score the same 3 hotspots.

In [37]:
need_revision = not evaluation['all_pass'].all()
print(f'Revision needed: {need_revision}')

if need_revision:
    TEMPLATE_V2 = """You are an operations analyst for UrbanEats.

Restaurant   : {restaurant_name}
Delivery zone: {delivery_zone}
High cancel-risk rate: {high_risk_rate}
Average order value  : {avg_order_value}
Average delivery time: {avg_delivery_time}

Write EXACTLY two sentences for the regional manager. Strict rules:
1. Sentence 1 MUST start with this exact phrase: \"{restaurant_name} in the {delivery_zone} zone is a cancellation hotspot at {high_risk_rate} high cancel-risk.\"
2. Sentence 2 MUST start with \"Recommend\" and name ONE action from: increase rider allocation, review restaurant prep times, audit kitchen capacity, adjust dispatch routing, add staff during peak hours.
3. Banned words (do not use any form): seems, perhaps, might, appears, possibly, could, may.
4. Output ONLY the two sentences. No greeting, no sign-off, no markdown."""

    prompt_v2 = PromptTemplate(
        input_variables=['restaurant_name','delivery_zone','high_risk_rate',
                         'avg_order_value','avg_delivery_time'],
        template=TEMPLATE_V2,
    )
    chain_v2 = prompt_v2 | llm | StrOutputParser()

    revised_alerts = []
    for _, row in sample.iterrows():
        new_alert = chain_v2.invoke(format_inputs(row)).strip()
        revised_alerts.append(new_alert)

    revised = sample[['restaurant_name','delivery_zone']].copy().reset_index(drop=True)
    revised['alert_v1'] = sample['alert'].values
    revised['alert_v2'] = revised_alerts
    rescored = revised.apply(
        lambda r: score_alert(r['alert_v2'], r['restaurant_name'], r['delivery_zone']),
        axis=1, result_type='expand'
    )
    rescored.columns = ['specificity_v2','actionability_v2','no_hedging_v2']
    revised = pd.concat([revised, rescored], axis=1)
    revised['all_pass_v2'] = revised[['specificity_v2','actionability_v2','no_hedging_v2']].all(axis=1)
    display(revised)
else:
    print('All 3 alerts passed all 3 criteria on the v1 prompt - no revision needed.')

Revision needed: False
All 3 alerts passed all 3 criteria on the v1 prompt - no revision needed.


## Cell 11 - Save final alerts to Drive

In [38]:
import os
from datetime import date

output_dir = os.path.dirname(SCORED_CSV_PATH)

# 1. Save the hotspot alerts (with date column)
alerts_path = os.path.join(output_dir, 'urbaneats_ops_alerts.csv')
hotspots.to_csv(alerts_path, index=False)
print(f'Saved {len(hotspots)} alerts to: {alerts_path}')

# 2. Build daily brief input for n8n: full scored data + alert text for hotspot rows
brief = scored.merge(
    hotspots[['restaurant_name', 'delivery_zone', 'high_risk_rate', 'alert']],
    on=['restaurant_name', 'delivery_zone'],
    how='left',
)
brief['alert']          = brief['alert'].fillna('')
brief['high_risk_rate'] = brief['high_risk_rate'].fillna(0.0)
brief['generated_at']   = date.today().isoformat()

brief_path = os.path.join(output_dir, 'urbaneats_daily_brief_input.csv')
brief.to_csv(brief_path, index=False)
print(f'Saved {len(brief)} brief-input rows to: {brief_path}')


Saved 13 alerts to: /content/drive/MyDrive/Colab Notebooks/urbaneats_ops_alerts.csv
Saved 150 brief-input rows to: /content/drive/MyDrive/Colab Notebooks/urbaneats_daily_brief_input.csv
